In [ ]:
'''
#Enable access to internal google resources 
gcloud compute networks subnets update default --region=REGION_NAME     --enable-private-ip-google-access

#Role reguired for VM service account to process tasks in Claster
gcloud projects get-iam-policy project-f2341c77-fecc-4b52-a12 \
--flatten="bindings[].members" \
--format='table(bindings.role)' \
--filter="bindings.members:767007221215-compute@developer.gserviceaccount.com"

# ROLE
# roles/dataproc.worker

#Role required to Dataproc service account
gcloud projects get-iam-policy project-f2341c77-fecc-4b52-a12 \
--flatten="bindings[].members" \
--filter="bindings.members:service-767007221215@dataproc-accounts.iam.gserviceaccount.com" \
--format="table(bindings.role)"

# ROLE
# roles/dataproc.serviceAgent

#Services to be enabled
gcloud services list \
--enabled \
--project=project-f2341c77-fecc-4b52-a12 \
--filter="name:cloudresourcemanager.googleapis.com" 

# NAME                                 TITLE
# cloudresourcemanager.googleapis.com  Cloud Resource Manager API

# if not - enable
gcloud services enable cloudresourcemanager.googleapis.com   --project=project-f2341c77-fecc-4b52-a12

# Just in case check service account is binded to you
gcloud iam service-accounts get-iam-policy \
767007221215-compute@developer.gserviceaccount.com \
--project=project-f2341c77-fecc-4b52-a12

# bindings:
# - members:
#   - user:Maxim.Savrilov@gmail.com
#   role: roles/iam.serviceAccountUser
# etag: BwZY-93Q9Zo=
# version: 1

gcloud iam service-accounts add-iam-policy-binding \
767007221215-compute@developer.gserviceaccount.com \
--member="user:maxim.savrilov@gmail.com" \
--role="roles/iam.serviceAccountUser" \
--project=project-f2341c77-fecc-4b52-a12

# Then create your cluster, the single node as example (optional component DOCKER fails to install for now)
gcloud dataproc clusters create cluster-working \
--enable-component-gateway \
--region=europe-west4 \
--subnet=default \
--no-address \
--single-node \
--master-machine-type=e2-standard-4 \
--master-boot-disk-type=pd-balanced \
--master-boot-disk-size=100 \
--image-version=2.3-debian12 \
--optional-components=ICEBERG,DELTA,JUPYTER \
--scopes='https://www.googleapis.com/auth/cloud-platform' \
--project=project-f2341c77-fecc-4b52-a12 \
--async

# For Docker it is required to create NAT router
gcloud compute routers create dataproc-router \
  --network=default \
  --region=europe-west4 \
  --project=project-f2341c77-fecc-4b52-a12
gcloud compute routers nats create dataproc-nat \
  --router=dataproc-router \
  --region=europe-west4 \
  --nat-all-subnet-ip-ranges \
  --auto-allocate-nat-external-ips \
  --project=project-f2341c77-fecc-4b52-a12
Then use "cloud dataproc jobs submit pyspark" to send jobs to your claster
'''

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [2]:
# 1. Привязываем правильный, стабильный shaded-джарник
conf = SparkConf() \
    .setMaster('spark://127.0.0.1:7077') \
    .setAppName('test') \
    .set("spark.jars", "./lib/gcs-connector-4.0.4-shaded.jar")

sc = SparkContext(conf=conf)

# 2. Прописываем оригинальные классы Google для работы с протоколом gs://
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")

hadoop_conf.set("fs.gs.block.size", "67108864")

26/08/13 12:07:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
# 3. Стартуем стандартную сессию Spark SQL (без отключения векторизации!)
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [5]:
spark

In [6]:
df_green = spark.read.option("recursiveFileLookup", "true").parquet('gs://maksim-savrilov-spark/pq/green/')

In [8]:
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [9]:
df_yellow = spark.read.option("recursiveFileLookup", "true").parquet('gs://maksim-savrilov-spark/pq/yellow/')

In [10]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [11]:
common_colums = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_colums.append(col)

In [12]:
from pyspark.sql import functions as F

In [13]:
df_green_sel = df_green \
    .select(common_colums) \
    .withColumn('service_type', F.lit('green'))

In [14]:
df_yellow_sel = df_yellow \
    .select(common_colums) \
    .withColumn('service_type', F.lit('yellow'))

In [15]:
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [16]:
df_trips_data.groupBy('service_type').count().show()

[Stage 4:=====================================================>   (15 + 1) / 16]

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2232260|
|      yellow|39649199|
+------------+--------+



In [17]:
df_trips_data.columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge',
 'service_type']

In [18]:
df_trips_data.createOrReplaceTempView('trips_data')

In [19]:
spark.sql("""
SELECT
    service_type,
    count(1)
FROM
    trips_data
GROUP BY 
    service_type
""").show()

[Stage 7:=====================================================>   (15 + 1) / 16]

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green| 2232260|
|      yellow|39649199|
+------------+--------+



In [20]:
df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [21]:
df_result.show()

[Stage 10:====================================================>   (15 + 1) / 16]

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_passenger_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|         250

In [22]:
df_result.coalesce(1).write.parquet('gs://maksim-savrilov-spark/report/revenue/', mode='overwrite')

In [23]:
df_result.limit(5).toPandas()

,revenue_zone,revenue_month,service_type,revenue_monthly_fare,revenue_monthly_extra,revenue_monthly_mta_tax,revenue_monthly_tip_amount,revenue_monthly_tolls_amount,revenue_monthly_improvement_surcharge,revenue_monthly_total_amount,revenue_monthly_congestion_surcharge,avg_monthly_passenger_count,avg_monthly_trip_distance
0,250,2020-02-01,green,15359.96,1282.50,117.5,56.01,590.32,180.0,17598.44,11.0,1.239496,4.962811
1,158,2020-02-01,green,124.36,8.25,0.5,0.00,2.80,0.9,136.81,NaN,NaN,11.090000
2,15,2020-03-01,green,1682.23,5.50,6.5,0.00,79.56,19.8,1802.44,0.0,1.000000,7.910299
3,229,2020-03-01,green,676.36,0.00,1.0,0.00,42.84,8.1,728.30,0.0,1.000000,7.691111
4,137,2020-10-01,green,3480.83,0.00,0.0,297.00,250.92,32.4,4061.15,NaN,NaN,9.184722
